# Classficação de Imagens (Intro) em um Raspberry Pi com TFLite  
 Fundamentos de Classificação de Imagens - MobileNet V2

> Adaptado da seção [*Image Classification Fundamentals*](https://mjrovai.github.io/EdgeML_Made_Ease_ebook/raspi/image_classification/image_classification_fund.html) do [Prof. Marcelo Rovai](https://github.com/Mjrovai) no livro [*EdgeML Made Easy*](https://mjrovai.github.io/EdgeML_Made_Ease_ebook/) e do repositório do GitHub [Edge Machine Learning Systems Engineering](https://github.com/Mjrovai/UNIFEI-IESTI05-EDGE_AI/tree/main).



## Introdução

Este notebook demonstra, como realizar classificação de imagens no Raspberry Pi usando um modelo TFLite (MobileNet V2 quantizado). O fluxo cobre desde o download dos recursos até a captura de imagens pela câmera e a exibição das previsões.

Principais objetivos
- Mostrar como baixar e preparar imagens e o modelo TFLite.
- Demonstrar pré-processamento (redimensionamento para 224×224 e batch).
- Executar inferência com tflite-runtime, desquantizar a saída e aplicar Softmax.
- Exibir resultados e criar uma função reutilizável de classificação.
- Capturar imagens com Picamera2 e classificar em tempo real.

Requisitos
- Se você ainda não instalou o TensorFlow Lite, siga as instruções na seção "Instalando o TensorFlow Lite no Raspberry Pi" do roteiro de laboratório [Instalação de Bibliotecas Python para o RPi](https://github.com/fabiobento/sis-emb-2025-2/blob/main/aulas/sbc-rpi/rpi_ei_linux_sdk/rpi_ei_linux_sdk.md)
- Biblioteca Pillow, NumPy, Matplotlib e Picamera2 (para captura).
- Arquivos: modelo TFLite (mobilenet_v2_1.0_224_quant.tflite) e labels.txt na pasta ./models; imagens em ./imagens.

Estrutura do notebook
1. Introdução e referências.
2. Download e extração de imagens e do modelo.
3. Importação de bibliotecas e configuração de warnings.
4. Funções utilitárias (ex.: load_labels).
5. Carregamento do modelo TFLite e inspeção dos tensores.
6. Carregamento e visualização das imagens de teste.
7. Pré-processamento, inferência, desquantização e aplicação de Softmax.
8. Função genérica image_classification para reutilização.
9. Testes com várias imagens e integração com Picamera2 para captura ao vivo.

Como usar
- Execute sequencialmente as células para garantir que variáveis e imports estejam disponíveis.
- Use a função image_classification(img_path, model_path, labels, top_k_results) para avaliar novas imagens.
- Para classificar a partir da câmera, execute a célula que chama capture_image() seguida de image_classification().

Observações
- O modelo quantizado utiliza entradas uint8; o código trata da desquantização e conversão para probabilidades.
- Atenção às permissões de hardware ao usar Picamera2 no Raspberry Pi.

## Importação de dados

Baixar imagens de teste para o Raspberry Pi:

In [ ]:
# Instala o gdown se não estiver instalado
!pip install -q gdown

# Baixa o arquivo do Google Drive
!gdown --id 1QuG15ERYLv7M5qULAzeEzDa5KVk7qL9I -O ./imagens/imagens_teste_drive.tar.xz

# Descompacta o arquivo baixado para a pasta ./imagens
!tar -xJf ./imagens/imagens_teste_drive.tar.xz -C ./imagens

# Lista o conteúdo da pasta para verificar a extração
!ls ./imagens

# Remove o arquivo compactado para economizar espaço
!rm ./imagens/imagens_teste_drive.tar.xz

In [ ]:
from pathlib import Path
import tarfile
import urllib.request

# Cria o diretório ./models se ele não existir (inclui diretórios pai)
models_dir = Path("./models")
models_dir.mkdir(parents=True, exist_ok=True)

# Define caminhos para o arquivo .tflite e para o arquivo compactado (.tgz)
tflite_file = models_dir / "mobilenet_v2_1.0_224_quant.tflite"
tgz_file = models_dir / "mobilenet_v2_1.0_224_quant.tgz"

# URL de onde baixar o modelo compactado
url = "https://storage.googleapis.com/download.tensorflow.org/models/tflite_11_05_08/mobilenet_v2_1.0_224_quant.tgz"

# Se o arquivo .tflite já existir, não faz download nem extração
if tflite_file.exists():
    print(f"{tflite_file} já existe. Pulando download e extração.")
else:
    # Se o arquivo .tgz ainda não foi baixado, baixa-o
    if not tgz_file.exists():
        print(f"Baixando {url} para {tgz_file} ...")
        urllib.request.urlretrieve(url, tgz_file)
    else:
        print(f"{tgz_file} já existe. Pulando download.")
    try:
        # Tenta extrair o conteúdo do arquivo .tgz para a pasta models
        print(f"Extraindo {tgz_file} para {models_dir} ...")
        with tarfile.open(tgz_file, "r:gz") as tar:
            tar.extractall(path=models_dir)
        print("Extração concluída.")
    except Exception as e:
        # Em caso de erro na extração, imprime a exceção
        print("Falha ao extrair:", e)

# Verifica se o arquivo .tflite está disponível após a extração
if tflite_file.exists():
    print("Arquivo .tflite pronto:", tflite_file)
else:
    print("Arquivo .tflite não encontrado após extração.")

## Importação de bibliotecas

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tflite_runtime.interpreter as tflite

- A célula abaixo importa o módulo warnings e configura o filtro para ignorar avisos do tipo UserWarning.  
  - Objetivo: reduzir ruído na saída do notebook, evitando que avisos não críticos apareçam durante a execução.  
  - Observação: usar com cautela — ocultar avisos pode mascarar problemas úteis para depuração. Para reativar avisos, remova a linha ou use warnings.filterwarnings('default').

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

## Carregamento do modelo TFLite e inspeção de tensores de entrada/saída

- Agora definimos uma função chamada `load_labels`, responsável por carregar os rótulos das classes a partir de um arquivo de texto.
- Essa função:
    - lê cada linha do arquivo especificado,
    - remove espaços em branco extras e
    - retorna uma lista contendo todos os rótulos. 
- Essa lista será utilizada posteriormente para associar os índices das previsões do modelo aos nomes das classes correspondentes, facilitando a interpretação dos resultados da classificação de imagens.

In [ ]:
# Função para carregar rótulos
def load_labels(filename):
    with open(filename, 'r') as f:
        return [line.strip() for line in f.readlines()]

Na célula abaixo vamos definir os caminhos dos arquivos necessários para a classificação de imagens:
- `model_path`: Caminho para o modelo TFLite pré-treinado MobileNet V2
- `img_path`: Caminho para a imagem de teste (gato)
- `labels_path`: Caminho para o arquivo com os rótulos das classes

In [ ]:
model_path = "./models/mobilenet_v2_1.0_224_quant.tflite"
img_path = "./imagens/Cat03.jpg"
labels_path = "./models/labels.txt"

Em seguida:
- carregamos o modelo TFLite MobileNet V2,
- inicializamos o interpretador e
- obtemos os detalhes dos tensores de entrada e saída necessários para realizar a inferência.

Esses detalhes ajudam a garantir que:
- a imagem de entrada seja processada corretamente e que
- os resultados da classificação possam ser interpretados.

In [ ]:
# Carregar o modelo TFLite
interpreter = tflite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# Obter tensores de entrada e saída
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [ ]:
input_details

In [ ]:
output_details

## Carregamento e preprocessamento da imagem de teste

- A célula de código abaixo realiza o carregamento de uma imagem a partir do caminho especificado na variável `img_path` utilizando a biblioteca PIL (`Image.open`).
- Assim a imagem pode ser:
    - visualizada,
    - processada e
    - posteriormente utilizada como entrada no modelo de classificação.
    
- O objeto retornado representa a imagem em formato manipulável pelo Python, permitindo operações como redimensionamento, exibição e transformação dos dados para o formato esperado pelo modelo de machine learning.

In [ ]:
# Carregar a imagem
img = Image.open(img_path)

- A célula de código abaixo exibe a imagem carregada anteriormente utilizando a biblioteca Matplotlib.

 Ela cria uma figura com tamanho definido, mostra a imagem na tela e adiciona um título ("Imagem Original") ao gráfico.
 
 - Com isso podemos visualizar a imagem original antes de realizar qualquer pré-processamento ou classificação
    - Nessa etapa você pode conferir se o carregamento foi feito corretamente e se a imagem está adequada para os próximos passos do fluxo de trabalho.

In [ ]:
# Exibir a imagem
plt.figure(figsize=(6, 6))
plt.imshow(img)
#plt.axis('off')  # Isso "desliga os números nos eixos"
plt.title("Imagem Original")
plt.show()

A célula de código abaixo:
- obtém as dimensões da imagem carregada (largura, altura e número de canais de cor), e
- armazena essas informações na variável `shape`.
- Em seguida, exibe o formato da imagem no console

Assim você pode confirmar que a imagem foi carregada corretamente e para verificar se suas dimensões e canais estão adequados para o processamento posterior no modelo de classificação.

In [ ]:
width, height = img.size
channels = len(img.getbands())
shape = (height, width, channels)

print(f"Formato da imagem: {shape}")

Agora realizamos o **pré-processamento** da imagem carregada:
- redimensionando-a para o tamanho esperado pelo modelo (224x224 pixels) e
- adicionando uma dimensão extra para representar o batch.

O resultado é um array pronto para ser utilizado como entrada na rede neural durante a inferência.

In [ ]:
# Preprocessar a imagem
img = img.resize((input_details[0]['shape'][1], input_details[0]['shape'][2]))
input_data = np.expand_dims(img, axis=0)
input_data.shape

A próxima célula exibe o tipo de dado (dtype) do array `input_data`.
- Assim você pode verificar se o formato da entrada coincide com o esperado pelo modelo TFLite (por exemplo `uint8` em modelos quantizados).
    - Se houver discrepância, é necessário converter (`.astype(...)`) antes de passar os dados ao interpretador.

In [ ]:
# Obter o tipo dos dados de entrada
input_data.dtype

A célula abaixo verifica o tipo de dado **esperado pelo modelo TFLite para a entrada**, acessando a propriedade `'dtype'` do tensor de entrada.

Isso garante que o array de imagem pré-processado (`input_data`) está no formato correto antes de ser passado para o interpretador, evitando erros de compatibilidade e facilitando o fluxo de inferência.

In [ ]:
# Obter o tipo dos dados de saída
input_dtype = input_details[0]['dtype']
input_dtype

O tipo de dados de entrada é `uint8`, que é compatível com o tipo de dados esperado para o modelo.

In [ ]:
# Exibir a imagem redimensionada
plt.figure(figsize=(5, 5))
plt.imshow(img)
#plt.axis('off')  # This turns off the axis numbers
plt.title("Resized Image (224x224)")
plt.show()

## Inferência, desquantização e aplicação de Softmax

### Inferência

A próxima célula faz a inferência do modelo TFLite e mede o tempo gasto:
- Inicia um cronômetro, define os dados de entrada no tensor do interpretador e chama `interpreter.invoke()` para executar a inferência.
- Calcula o tempo total de inferência em milissegundos e o imprime formatado.
- OBS: depende de variáveis/objetos previamente definidos (por exemplo, `interpreter`, `input_details` e `input_data`) e fornece a métrica de desempenho usada nas células seguintes.

In [ ]:
# Realiza a inferência no Raspberry Pi
start_time = time.time()  # Inicia a contagem do tempo
interpreter.set_tensor(input_details[0]['index'], input_data)  # Define os dados de entrada para o modelo
interpreter.invoke()  # Executa a inferência
end_time = time.time()  # Finaliza a contagem do tempo
inference_time = (end_time - start_time) * 1000  # Converte o tempo para milissegundos
print("Tempo de inferência: {:.1f}ms".format(inference_time))  # Exibe o tempo de inferência

- Agora recuperamos do interpretador TFLite o tensor de saída gerado pela última inferência.
    - Usamos `output_details[0]['index']` para acessar o índice do tensor de saída e `[0]` para remover a dimensão de batch (ficando apenas o vetor de pontuações por classe).
- OBS: Depende de `interpreter` e `output_details` já inicializados e de `interpreter.invoke()` ter sido executado anteriormente.
- Resultado: um array 1D com as previsões quantizadas (valores brutos) que serão desquantizadas e convertidas em probabilidades nas células seguintes.

In [ ]:
# Obter as previsões e mapear para rótulos
predictions = interpreter.get_tensor(output_details[0]['index'])[0]

A célula de código abaixo exibe o vetor de previsões gerado pelo modelo TFLite após a inferência.  
- Esse vetor contém os valores brutos (quantizados) para cada classe possível, representando o grau de correspondência da imagem analisada com cada categoria do modelo.  
- Esses valores ainda não são probabilidades e precisam ser desquantizados e normalizados (softmax) para interpretação final, como será feito nas próximas etapas do notebook.

In [ ]:
predictions

Exibir o formato (shape) do vetor de previsões gerado pelo modelo TFLite.  
- Isso permite verificar quantas classes o modelo está considerando e garante que o processamento posterior (seleção dos melhores resultados, desquantização e aplicação do softmax) será feito sobre o número correto de categorias.  

- Com isso, validamos que a saída da inferência está conforme o esperado antes de prosseguir para a interpretação dos resultados.

In [ ]:
predictions.shape

A célula de código abaixo seleciona os índices dos **top 5 resultados** da classificação, ou seja, as classes que receberam as maiores pontuações do modelo para a imagem analisada.  
- Utiliza a função `np.argsort` para ordenar as previsões em ordem decrescente e extrai os índices correspondentes às classes mais prováveis.  
    - Esses índices serão usados para exibir os nomes das classes e suas probabilidades nas etapas seguintes, facilitando a interpretação dos resultados da inferência.

In [ ]:
# Obter os rótulos dos k melhores resultados
top_k_results = 5
top_k_indices = np.argsort(predictions)[::-1][:top_k_results]
top_k_indices

Em seguida carregamos os rótulos das classes do modelo a partir do arquivo `labels.txt`, utilizando a função `load_labels`.
- Com esses rótulos interpretamos os resultados da classificação, pois eles permitem associar cada índice de saída do modelo ao nome correspondente da classe prevista.
- Assim, ao realizar a inferência, será possível exibir os nomes das categorias identificadas na imagem, tornando o resultado compreensível e útil para o usuário.

In [ ]:
# Carregar os rótulos
labels = load_labels(labels_path)

Vamos inprimir os nomes das classes correspondentes aos índices dos principais resultados da classificação, bem como os valores brutos (quantizados) das previsões para cada uma dessas classes.

In [ ]:
print(labels[286])
print(labels[283])
print(labels[282])
print(labels[288])
print(labels[479])

Na próxima célula imprimiremos os valores quantizados (*raw*) das previsões para as 5 classes mais relevantes identificadas pelo modelo.

Esses valores ainda não foram convertidos para probabilidades (0-1) e precisam passar por desquantização e normalização via softmax para representarem as probabilidades finais de cada classe.


In [ ]:
print (predictions[286])
print (predictions[283])
print (predictions[282])
print (predictions[288])
print (predictions[479])

### Desquantização

- Agora vamos extrair os parâmetros de quantização (`scale` e `zero_point`) do tensor de saída do interpretador `TFLite`.
    - Em modelos quantizados (`uint8`), a saída do modelo é um valor inteiro; para obter os valores reais precisamos desquantizar usando:
    real = (quantizado - zero_point) * scale
- Esses parâmetros são essenciais para converter as previsões brutas em `floats` antes de aplicar o `Softmax` e obter probabilidades interpretáveis.

Esta célula prepara exatamente esses valores para a etapa seguinte.

In [ ]:
# Obter os parâmetros de quantização da saída
scale, zero_point = output_details[0]['quantization']
scale, zero_point

Vamos fazer agora a **desquantização** dos valores de saída do modelo:
- Converte as previsões brutas (inteiros 0-255) em valores reais (`float`) usando os parâmetros `scale` e `zero_point` obtidos anteriormente
- Aplica a fórmula: `real = (quantizado - zero_point) * scale`
- O resultado são valores não normalizados que precisarão passar por `softmax` para representar probabilidades

In [ ]:
# Desquantizar as previsões
dequantized_output = (predictions.astype(np.float32) - zero_point) * scale
dequantized_output

### Aplicação de Softmax

- A saída (números positivos e negativos) mostra que a saída provavelmente não tem um *Softmax*.
    - Verificando a documentação do modelo (https://arxiv.org/abs/1801.04381v4): o MobileNet V2 normalmente não inclui uma camada `softmax` na saída.
        - Ele geralmente termina com uma convolução 1x1 seguida por *pooling* médio e uma camada totalmente conectada.
- Portanto, para obter as probabilidades (0 a 1), devemos aplicar o [`Softmax`](https://en.wikipedia.org/wiki/Softmax_function):

In [ ]:
# Aplicar softmax
exp_output = np.exp(dequantized_output - np.max(dequantized_output))
probabilities = exp_output / np.sum(exp_output)

A célula abaixo exibe o vetor `probabilities` — ou seja, as probabilidades normalizadas (valores em 0–1) para cada classe, obtidas após desquantizar a saída quantizada do modelo e aplicar Softmax. Pontos-chave:

- Cada elemento do vetor representa a probabilidade da imagem pertencer à respectiva classe do modelo.  
- Os valores somam aproximadamente 1 (verificação na célula seguinte).  
- Para obter percentuais, multiplica-se por 100 (as impressões seguintes mostram os top-k como porcentagem).  
- As células subsequentes usam `top_k_indices` para mapear essas probabilidades aos rótulos (`labels`) e apresentar os principais resultados formatados.

Essa etapa finaliza a conversão das saídas brutas do TFLite em probabilidades interpretáveis, prontas para serem exibidas ao usuário.

In [ ]:
probabilities 

In [ ]:
# Se somarmos todos os valores, podemos obter cerca de 1.
probabilities.sum()

Agora vamos verificar os valores das probabilidades para as 5 classes principais identificadas pelo modelo.
- Após a desquantização e aplicação do Softmax, os valores estão normalizados entre 0 e 1
- Estes números representam a confiança do modelo em cada previsão
- Valores mais próximos de 1 indicam maior certeza na classificação

In [ ]:
print (probabilities[286])
print (probabilities[283])
print (probabilities[282])
print (probabilities[288])
print (probabilities[479])

A célula abaixo exibe os resultados finais da classificação formatados, mostrando:
- Os nomes das classes (rótulos) correspondentes aos 5 índices com maior probabilidade
- A probabilidade de cada classe em formato percentual (0-100%)
- O alinhamento e formatação facilitam a leitura dos resultados principais


In [ ]:
for i in range(top_k_results):
    print("\t{:20}: {}%".format(
        labels[top_k_indices[i]],
        (int(probabilities[top_k_indices[i]]*100))))

## Uma função mais geral de Classificação de Imagens

A célula define uma função reutilizável `image_classification(img_path, model_path, labels, top_k_results=5)` que realiza todo o fluxo de inferência para uma imagem:
  1. carrega e exibe a imagem
  2. inicializa o interpretador TFLite
  3. obtém os tensores
  4. redimensiona a imagem ao tamanho de entrada do modelo e
  5. insere o batch
  6. executa a inferência
  7. recupera as previsões quantizadas
  8. desquantiza usando `scale` e `zero_point`
  9. aplica softmax para obter probabilidades
  10. imprime os `top_k_results` rótulos com suas probabilidades (em %).  

In [ ]:
def image_classification(img_path, model_path, labels, top_k_results=5):
    # Carregar a imagem
    img = Image.open(img_path)
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis('off')  # esconder eixos

    # Carregar o modelo TFLite
    interpreter = tflite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    
    # Obter tensores de entrada e saída
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Pré-processamento: redimensionar para o tamanho esperado pelo modelo
    img = img.resize((input_details[0]['shape'][1], 
                      input_details[0]['shape'][2]))
    # Adicionar dimensão de batch
    input_data = np.expand_dims(img, axis=0)

    # Inferência
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    
    # Obter previsões brutas (quantizadas)
    predictions = interpreter.get_tensor(output_details[0]['index'])[0]

    # Índices dos top k resultados
    top_k_indices = np.argsort(predictions)[::-1][:top_k_results]

    # Parâmetros de quantização (scale e zero_point)
    scale, zero_point = output_details[0]['quantization']

    # Desquantizar a saída e aplicar softmax para obter probabilidades
    dequantized_output = (predictions.astype(np.float32) - zero_point) * scale
    exp_output = np.exp(dequantized_output - np.max(dequantized_output))
    probabilities = exp_output / np.sum(exp_output)

    # Imprimir os resultados formatados
    print("\n\t[PREDIÇÃO]           [Prob]\n")
    for i in range(top_k_results):
        print("\t{:20}: {}%".format(
            labels[top_k_indices[i]],
            (int(probabilities[top_k_indices[i]]*100))))

Abaixo definimos os caminhos dos arquivos e preparamos os rótulos para a classificação. Especificamente:
- atribuimos às variáveis `model_path`, `img_path` e `labels_path` os caminhos para o modelo TFLite, a imagem de teste e o arquivo de rótulos, respectivamente;
- carregamos os rótulos com a função `load_labels`, produzindo a lista `labels` que mapeia índices do modelo para nomes legíveis.

Execute esta célula antes de chamar `image_classification(...)`.

Para testar outras imagens, altere `img_path` e reexecute a célula seguida da chamada de inferência.

In [ ]:
model_path = "./models/mobilenet_v2_1.0_224_quant.tflite"
img_path = "./imagens/Cat03.jpg"
labels_path = "./models/labels.txt"
labels = load_labels(labels_path)

In [ ]:
image_classification(img_path, model_path, labels, top_k_results=5)

In [ ]:
!ls ./imagens

In [ ]:
img_path = "./imagens/car_1.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/car_2.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/car_3.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/car_4.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/car_5.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/cat_1.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/cat_2.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/dog_1.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/dog_2.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/dog_3.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/ship_1.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/ship_2.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
ls

## Classificando Imagens da Câmera

- Definimos agora uma função chamada `capture_image` que utiliza a biblioteca `picamera2` para controlar uma câmera do Raspberry Pi.

    - O objetivo da função é tirar uma única foto com configurações específicas e salvá-la em um arquivo.

- De forma simplificada, o processo ocorre da seguinte maneira:
    1. Inicialização: Primeiro, a câmera é "ligada" e preparada para uso com o comando `Picamera2()`.

    2. Configuração: Em seguida, a câmera é configurada para capturar uma imagem estática com uma resolução exata de 224x224 pixels. Este tamanho específico foi escolhido para compatibilidade com o nosso modelo de visão computacional(Mobilinet V2), que espera imagens com dimensões pré-definidas.

    3. Aquecimento: O código faz uma pausa de 2 segundos. Esse tempo é importante para que o sensor da câmera se estabilize e ajuste automaticamente o foco e a exposição à luz do ambiente, garantindo uma imagem de melhor qualidade.

    4. Captura: Com a câmera pronta, o comando capture_file tira a foto e a salva diretamente no caminho de arquivo fornecido (a variável `image_path`).

    5. Desligamento: Por fim, a função desliga a câmera e libera os recursos do sistema com os comandos `stop()` e `close()`. Isso garante que a câmera esteja disponível para ser usada novamente por outros programas.

In [ ]:
from picamera2 import Picamera2

def capture_image(image_path):
  # Inicializa a câmera Picamera2 (usa o índice padrão 0)
  picam2 = Picamera2()

  # Configura a câmera para captura de imagem estática no tamanho esperado pelo modelo (224x224)
  config = picam2.create_still_configuration(main={"size": (224, 224)})
  picam2.configure(config)
  picam2.start()

  # Aguarda a câmera 'aquecer' / estabilizar
  time.sleep(2)

  # Captura a imagem e salva no caminho especificado
  picam2.capture_file(image_path)
  print(f"Imagem capturada: {image_path}")

  # Para a câmera e libera recursos
  picam2.stop()
  picam2.close()

E, finalmente, chamamos a função `capture_image` para tirar uma foto e salvá-la no caminho especificado por `img_path`.

Este bloco de código executa um processo completo de classificação de imagem em tempo real com os seguintes passos:
1. Preparação: Primeiro, o código define os caminhos para os arquivos essenciais: onde a imagem capturada pela câmera será salva (img_path), a localização do modelo de inteligência artificial treinado (model_path), e o arquivo de texto que contém os nomes de todos os objetos que o modelo consegue reconhecer (labels).
2. Captura: Em seguida, ele chama a função `capture_image()` para usar a câmera, tirar uma foto e salvá-la no local definido.
3. Classificação: Finalmente, com a imagem recém-capturada em mãos, ele aciona a função `image_classification()`. Essa função carrega a imagem, utiliza o modelo mobilenet para analisá-la e, com base nos labels, exibe os 5 resultados mais prováveis do que o modelo "acredita" estar na foto.

In [ ]:
# Define o caminho onde a imagem capturada pela câmera será salva.
img_path = './imagens/cam_img_test.jpg'

# Define o caminho para o arquivo do modelo de machine learning (TensorFlow Lite).
model_path = "./models/mobilenet_v2_1.0_224_quant.tflite"

# Carrega a lista de rótulos (nomes dos objetos) que o modelo consegue reconhecer.
labels = load_labels("./models/labels.txt")

# Chama a função para tirar uma foto com a câmera e salvá-la no caminho definido acima.
capture_image(img_path)

# Executa a função de classificação. Ela usa o modelo para analisar a imagem capturada
# e exibe os 5 resultados mais prováveis com base nos rótulos carregados.
image_classification(img_path, model_path, labels, top_k_results=5)